<a href="https://colab.research.google.com/github/esrakirbas/msc-thesis-reducing-hallucinations-in-medical-diagnosis/blob/main/experiments/experiment01_medqa_manual_grounding/Medical_Grounding_001_Data_Preparation_questions_and_corpus_from_MedQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets

In [ ]:
from huggingface_hub import HfApi
info = HfApi().dataset_info("awinml/medqa") #get current revision number of MedQA (afdb980b627374a7f10a0a11ed68455596c52742)
print(info.sha)

afdb980b627374a7f10a0a11ed68455596c52742


In [ ]:
from datasets import load_dataset
questions = load_dataset("awinml/medqa", "questions", revision="afdb980b627374a7f10a0a11ed68455596c52742")
print(questions)
#to explore the first 20 questions (I will select some diagnosis questions among them. SELECTED indices : [2, 6, 15, 19] )
for i, item in enumerate(questions["test"].select(range(20))):
    print(f"\nQuestion {i+1} (index {i}):")
    print(item["question"])
    print(item["options"])
    print(item["answer"])
    print("-" * 60)

README.md:   0%|          | 0.00/8.98k [00:00<?, ?B/s]

questions/train.jsonl: reconstructing file:   0%|          |  0.00B / 16.2MB            

questions/train.jsonl: downloading bytes:           |  0.00B            

validation.jsonl:   0%|          | 0.00/2.02M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/2.08M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases'],
        num_rows: 10178
    })
    validation: Dataset({
        features: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases'],
        num_rows: 1272
    })
    test: Dataset({
        features: ['question', 'answer', 'options', 'meta_info', 'answer_idx', 'metamap_phrases'],
        num_rows: 1273
    })
})

Question 1 (index 0):
A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication o

In [ ]:
import re #built in regular exp. lib.
from datasets import load_dataset
#load manually selected questions
questions = load_dataset("awinml/medqa", "questions", revision="afdb980b627374a7f10a0a11ed68455596c52742")
selected_question_indices = [2, 6, 15, 19]
selected_questions = questions["test"].select(selected_question_indices)
#load manually selected textbooks
books = load_dataset("awinml/medqa", "corpus", revision="afdb980b627374a7f10a0a11ed68455596c52742")
selected_textbook_indices = [8, 17]
selected_textbooks = books["train"].select(selected_textbook_indices)
book_InternalMed = selected_textbooks[0]
book_Surgery = selected_textbooks[1]
#try to find a related passage as a grounding context for the first question selected_index = 2
text = book_InternalMed["text"]
keyword = "ataxia" #for question 16 : "porcelain gallbladder" for question 7 : "urinary tract obstruction" , for question 3 : "atheroembolism"
matches = list(re.finditer(re.escape(keyword), text, re.IGNORECASE)) #escape means treat special chars as normal text
#to see the matched passages. After investigating I used those positions to get the related paragraphs
for match in matches: #commented out after finding the candidate grounding paragraph
    position = match.start()
    print("Position:", position)
    print(text[max(0, position-100):position+2000])

'''
grounding_contexts = [None] * len(selected_questions)
#Grounding Context for Question 3 - index:2
related_paragraph_start_position = text.find("Atheroembolism is another cause of limb ischemia. In this conditi")
if related_paragraph_start_position != -1:
  #related_paragraph_end_position = text[related_paragraph_start_position:].find("\n") + related_paragraph_start_position
  related_paragraph_end_position = text.find("\n", related_paragraph_start_position)
  if related_paragraph_end_position != -1:
    grounding_contexts[0] = text[related_paragraph_start_position:related_paragraph_end_position]
print("grounding_contexts[0] : ", grounding_contexts[0][:300])
#Grounding Context for Question 7 - index:6
chapter_start = text.find("urinary tract Obstruction Julian L. Seifter") #I found this from the above for loop, this is the begining of the related chapter
chapter_end = text.find("Approach to the Patient with Gastrointestinal Disease",chapter_start)
grounding_contexts[1] = text[chapter_start:chapter_end]
print("grounding_contexts[1] : ", grounding_contexts[1][:300])
#Grounding Context for Question 16 - index:15
chapter_start = text.lower().find("limey (milk of calcium) bile and porcelain gallbladder") #I found this from the above for loop, this is the begining of the related chapter
chapter_end = text.lower().find("carcinoma of the gallbladder.",chapter_start) + len("carcinoma of the gallbladder.")
grounding_contexts[2] = text[chapter_start:chapter_end]
print("grounding_contexts[2] : ", grounding_contexts[2][:300])
#Grounding Context for Question 20 - index:19 (2 paragraphs - founded grounding context provides partial evidence.)
chapter_start = text.lower().find("in striking contrast is the hypoactive subtype") #I found this from the above for loop, this is the begining of the related chapter
chapter_end = text.lower().find("with prominent apathy and psychomotor slowing.",chapter_start)+len("with prominent apathy and psychomotor slowing.")
grounding_contexts[3] = text[chapter_start:chapter_end]
chapter_start = text.lower().find("in contrast, in opioid and sedative-hypnotic poisoning") #I found this from the above for loop, this is the begining of the related chapter
chapter_end = text.lower().find("grade 3 or 4 physiologic depression; [table 473e-2]).",chapter_start) + len("grade 3 or 4 physiologic depression; [table 473e-2]).")
grounding_contexts[3] += "\n\n"
grounding_contexts[3] += text[chapter_start:chapter_end]
print("grounding_contexts[3] : ", grounding_contexts[3][:300]) '''

#print the grounding data to a txt
#with open("grounding_contexts.txt", "w", encoding="utf-8") as f:
#    for i, context in enumerate(grounding_contexts):
#        f.write(f"\n{'='*60}\n")
#        f.write(f"Grounding Context {i} - Question {selected_question_indices[i] + 1}\n")
#        f.write(f"{'='*60}\n")
#        f.write(context + "\n")
# After creating this file, I manually corrected some formatting errors,
# such as removing misplaced tables and page numbers.



Position: 1230525
 near the root entry zone of the auditory nerve. Symptoms such as double vision, numbness, and limb ataxia suggest a brainstem or cerebellar lesion.

Because dizziness and imbalance can be a manifestation of a variety of neurologic disorders, the neurologic examination is important in the evaluation of these patients. Particular focus should be given to assessment of eye movements, vestibular function, and hearing.

The range of eye movements and whether they are equal in each eye should be observed. Peripheral eye movement disorders (e.g., cranial neuropathies, eye muscle weakness) are usually disconjugate (different in the two eyes). One should check pursuit (the ability to follow a smoothly moving target) and saccades (the ability to look back and forth accurately between two targets). Poor pursuit or inaccurate (dysmetric) saccades usually indicates central pathology, often involving the cerebellum. Finally, one should look for spon taneous nystagmus, an involunta

In [ ]:
from datasets import load_dataset
books = load_dataset("awinml/medqa", "corpus", revision="afdb980b627374a7f10a0a11ed68455596c52742")
print(books)
#to explore the text books and find the relevant one's indexes. MANUALLY SELECTED TEXT BOOKS : [8]:InternalMed_Harrison and [17]:Surgery_Schwartz
for i, item in enumerate(books["train"]):
  print(f"\nText Book {i} : {item["title"]}")


DatasetDict({
    train: Dataset({
        features: ['doc_id', 'title', 'source_filename', 'text'],
        num_rows: 18
    })
})

Text Book 0 : Anatomy_Gray

Text Book 1 : Biochemistry_Lippincott

Text Book 2 : Cell_Biology_Alberts

Text Book 3 : First_Aid_Step1

Text Book 4 : First_Aid_Step2

Text Book 5 : Gynecology_Novak

Text Book 6 : Histology_Ross

Text Book 7 : Immunology_Janeway

Text Book 8 : InternalMed_Harrison

Text Book 9 : Neurology_Adams

Text Book 10 : Obstentrics_Williams

Text Book 11 : Pathology_Robbins

Text Book 12 : Pathoma_Husain

Text Book 13 : Pediatrics_Nelson

Text Book 14 : Pharmacology_Katzung

Text Book 15 : Physiology_Levy

Text Book 16 : Psichiatry_DSM-5

Text Book 17 : Surgery_Schwartz


In [ ]:
from datasets import load_dataset
#load manually selected questions
questions = load_dataset("awinml/medqa", "questions", revision="afdb980b627374a7f10a0a11ed68455596c52742")
selected_question_indices = [2, 6, 15, 19]
selected_questions = questions["test"].select(selected_question_indices)
#load manually selected textbooks
books = load_dataset("awinml/medqa", "corpus", revision="afdb980b627374a7f10a0a11ed68455596c52742")
selected_textbook_indices = [8, 17] #InternalMed_Harrison and Surgery_Schwartz
selected_textbooks = books["train"].select(selected_textbook_indices)
#to explore the structure of the books
print(selected_textbooks.column_names)
for i, book in enumerate(selected_textbooks):
  print("\nBook Title:", book["title"])
  print("Text length:", len(book["text"]))
  print("First 1000 characters:")
  print(book["text"][:1000])

['doc_id', 'title', 'source_filename', 'text']

Book Title: InternalMed_Harrison
Text length: 22312859
First 1000 characters:
xxxviii its related products in new formats. Kim Davis, as Associate Managing We are privileged to have compiled this 19th edition and areEditor, has adeptly ensured that the complex production of this multi-enthusiastic about all that it offers our readers. We learned much in the authored textbook proceeded smoothly and efficiently. Dominik Pucek process of editing Harrison’s and hope that you will find this edition a oversaw the production of the new procedural videos and Priscilla uniquely valuable educational resource. Beer expertly oversaw the production of our extensive DVD content. The Editors Jeffrey Herzich ably served as production manager for this new edition.

Chapter 1 The Practice of Medicine the practice of Medicine The Editors THE PHYSICIAN IN THE TWENTY-FIRST CENTURY No greater opportunity, responsibility, or obligation can fall to the lot of a 

In [ ]:
import json
records = []
#read grounding context from the related file
with open("grounding_contexts.txt", "r", encoding="utf-8") as file:
    all_contexts = file.read()
    for question_index in selected_question_indices:
      question = questions["test"][question_index]

      #read grounding context by question index
      search_start_keyword = "Question " + str(question_index + 1)
      search_end_keyword = "============================================================"
      start_index = all_contexts.find(search_start_keyword)
      if(start_index > -1):
          temp_str = all_contexts[start_index+len(search_start_keyword):]
          start_index = temp_str.find("\n")+1
          temp_str = temp_str[start_index:]
          start_index = temp_str.find("\n")+1
          if(start_index > -1):
              temp_str = temp_str[start_index:]
              end_index = temp_str.find(search_end_keyword)
              grounding_context = temp_str[:end_index].rstrip()
              question["options"] = {key: value.rstrip('\n"') for key, value in question["options"].items()}
              record = {"question_index":str(question_index), "question":question["question"], "options":question["options"], "answer":question["answer"].rstrip('\n"'), "grounding_context":grounding_context}
              records.append(record)
#create a json file and dump data in it.
with open("experiment01_questions_and_corpus.json", "w", encoding="utf-8") as file:
    json.dump(records, file, indent=2)

for rec in records:
  prompt = "Question Index " + rec["question_index"] + " : "
  prompt += rec["question"]
  prompt += "\n"
  for option, value in rec["options"].items():
    prompt += option + ". " + value + "\n"
  print(prompt)


Question Index 2 : Two weeks after undergoing an emergency cardiac catherization with stenting for unstable angina pectoris, a 61-year-old man has decreased urinary output and malaise. He has type 2 diabetes mellitus and osteoarthritis of the hips. Prior to admission, his medications were insulin and naproxen. He was also started on aspirin, clopidogrel, and metoprolol after the coronary intervention. His temperature is 38°C (100.4°F), pulse is 93/min, and blood pressure is 125/85 mm Hg. Examination shows mottled, reticulated purplish discoloration of the feet. Laboratory studies show:
Hemoglobin count 14 g/dL
Leukocyte count 16,400/mm3
Segmented neutrophils 56%
Eosinophils 11%
Lymphocytes 31%
Monocytes 2%
Platelet count 260,000/mm3
Erythrocyte sedimentation rate 68 mm/h
Serum
Urea nitrogen 25 mg/dL
Creatinine 4.2 mg/dL
Renal biopsy shows intravascular spindle-shaped vacuoles. Which of the following is the most likely cause of this patient's symptoms?"
A. Renal papillary necrosis
B. Ch